# RAMBHA-LP — MAVEN pretrain + transfer (Kaggle/Colab GPU)

Phase 3. Pretrain on Mars-like (denser) sweeps, then fine-tune on the lunar synthetic set.
Produces `maven_transfer.pt`, which the local `05_apply_and_recover.py` prefers automatically.

Works on either platform: set `SOL` to the uploaded/mounted `solution/` folder. This notebook
attempts a real MAVEN LPW reachability check (LASP archive) and otherwise uses the physics-
faithful MAVEN-like distribution — either way the transfer stage runs and is reported.

In [ ]:
import sys, os, glob
SOL = None
for c in glob.glob('/kaggle/input/**/config.yaml', recursive=True) + \
        glob.glob('/content/**/config.yaml', recursive=True):
    SOL = os.path.dirname(c); break
SOL = SOL or '/content/drive/MyDrive/RAMBHA/solution'
sys.path.insert(0, os.path.join(SOL, 'src'))
print('solution dir:', SOL)

In [ ]:
import torch, numpy as np, json, pathlib
from rambhalp.config import load_config
from rambhalp import maven, synthetic, train
cfg = load_config(os.path.join(SOL, 'config.yaml'))
cfg['train']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['maven']['pretrain_epochs'] = 120
cfg['train']['epochs'] = 120
print('device', cfg['train']['device'])

In [ ]:
# 1) Mars-like pretraining data (with a real-archive reachability probe)
Xpt, Ypt, mode = maven.get_pretrain_data(cfg, verbose=True)
print('MAVEN mode:', mode)
pre_model, _ = train.train_model(Xpt, Ypt, cfg, epochs=cfg['maven']['pretrain_epochs'])
pre_state = {k: v.clone() for k, v in pre_model.state_dict().items()}

In [ ]:
# 2) Fine-tune on lunar synthetic from the pretrained weights
Xtr, Ytr, _ = synthetic.make_dataset(cfg['synthetic']['n_train'], cfg, seed=1337)
Xva, Yva, _ = synthetic.make_dataset(cfg['synthetic']['n_val'], cfg, seed=1338)
model, hist = train.train_model(Xtr, Ytr, cfg, X_val=Xva, Y_val=Yva, init_state=pre_state)

In [ ]:
# 3) Save the transfer checkpoint
outdir = pathlib.Path('/kaggle/working') if os.path.exists('/kaggle/working') else \
         (pathlib.Path(SOL) / 'outputs' / 'checkpoints')
outdir.mkdir(parents=True, exist_ok=True)
train.save_checkpoint(model, cfg, outdir / 'maven_transfer.pt',
                      extra={'kind': 'maven_transfer', 'mode': mode, 'history': hist})
print('saved maven_transfer.pt to', outdir, '| mode', mode)